# Day 1 data profiling

Read-only profile of the Brazilian E-Commerce Public Dataset by Olist. Run this notebook from the repository root. Raw CSV files are loaded but never written back.

In [1]:
from pathlib import Path
import sys
import pandas as pd

REPO_ROOT = Path.cwd()
RAW_DIR = REPO_ROOT / 'data' / 'raw'
assert RAW_DIR.is_dir(), f'Run this notebook from the repository root: {RAW_DIR}'

print(f'Python: {sys.version.split()[0]}')
print(f'Pandas: {pd.__version__}')
print('Raw directory: data/raw')


Python: 3.13.7
Pandas: 3.0.6
Raw directory: data/raw


In [2]:
csv_paths = sorted(RAW_DIR.glob('*.csv'))
assert csv_paths, f'No CSV files found in {RAW_DIR}'
tables = {path.stem: pd.read_csv(path) for path in csv_paths}
print(f'Loaded {len(tables)} CSV files')
print(' | '.join(sorted(tables)))


Loaded 9 CSV files
olist_customers_dataset | olist_geolocation_dataset | olist_order_items_dataset | olist_order_payments_dataset | olist_order_reviews_dataset | olist_orders_dataset | olist_products_dataset | olist_sellers_dataset | product_category_name_translation


In [3]:
def date_columns(df):
    return [
        c for c in df.columns
        if any(token in c.lower() for token in ('date', 'timestamp')) or c.lower().endswith('_at')
    ]

profile_rows = []
for name, df in tables.items():
    missing = df.isna().sum()
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': len(df.columns),
        'full_row_duplicates': int(df.duplicated().sum()),
        'missing_cells': int(missing.sum()),
        'missing_columns': int((missing > 0).sum()),
    })
profile = pd.DataFrame(profile_rows).set_index('table')
display(profile)

print('Column names and dtypes')
for name, df in tables.items():
    print(f'\n{name}')
    display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'missing_pct': (df.isna().mean() * 100).round(2)}))

print('Date ranges')
date_range_rows = []
for name, df in tables.items():
    for col in date_columns(df):
        parsed = pd.to_datetime(df[col], errors='coerce')
        if parsed.notna().any():
            date_range_rows.append({'table': name, 'column': col, 'min': parsed.min(), 'max': parsed.max(), 'missing': int(parsed.isna().sum())})
display(pd.DataFrame(date_range_rows))

print('Numeric summaries')
numeric_rows = []
for name, df in tables.items():
    for col in df.select_dtypes(include='number').columns:
        s = df[col].dropna()
        numeric_rows.append({'table': name, 'column': col, 'count': int(s.size), 'min': s.min(), 'median': s.median(), 'mean': s.mean(), 'max': s.max()})
display(pd.DataFrame(numeric_rows).round(3))


,rows,columns,full_row_duplicates,missing_cells,missing_columns
table,,,,,
olist_customers_dataset,99441,5,0,0,0
olist_geolocation_dataset,1000163,5,261831,0,0
olist_order_items_dataset,112650,7,0,0,0
olist_order_payments_dataset,103886,5,0,0,0
olist_order_reviews_dataset,99224,7,0,145903,2
olist_orders_dataset,99441,8,0,4908,3
olist_products_dataset,32951,9,0,2448,8
olist_sellers_dataset,3095,4,0,0,0
product_category_name_translation,71,2,0,0,0


Column names and dtypes

olist_customers_dataset


,dtype,missing,missing_pct
customer_id,str,0,0.0
customer_unique_id,str,0,0.0
customer_zip_code_prefix,int64,0,0.0
customer_city,str,0,0.0
customer_state,str,0,0.0



olist_geolocation_dataset


,dtype,missing,missing_pct
geolocation_zip_code_prefix,int64,0,0.0
geolocation_lat,float64,0,0.0
geolocation_lng,float64,0,0.0
geolocation_city,str,0,0.0
geolocation_state,str,0,0.0


...

,dtype,missing,missing_pct
product_category_name,str,0,0.0
product_category_name_english,str,0,0.0


Date ranges


,table,column,min,max,missing
0,olist_order_items_dataset,shipping_limit_date,2016-09-19 00:15:34,2020-04-09 22:35:08,0
1,olist_order_reviews_dataset,review_creation_date,2016-10-02 00:00:00,2018-08-31 00:00:00,0
2,olist_order_reviews_dataset,review_answer_timestamp,2016-10-07 18:32:28,2018-10-29 12:27:35,0
3,olist_orders_dataset,order_purchase_timestamp,2016-09-04 21:15:19,2018-10-17 17:30:18,0
4,olist_orders_dataset,order_approved_at,2016-09-15 12:16:38,2018-09-03 17:40:06,160
5,olist_orders_dataset,order_delivered_carrier_date,2016-10-08 10:34:01,2018-09-11 19:48:28,1783
6,olist_orders_dataset,order_delivered_customer_date,2016-10-11 13:46:32,2018-10-17 13:22:46,2965
7,olist_orders_dataset,order_estimated_delivery_date,2016-09-30 00:00:00,2018-11-12 00:00:00,0


Numeric summaries


,table,column,count,min,median,mean,max
0,olist_customers_dataset,customer_zip_code_prefix,99441,1003.000,24416.000,35137.475,99990.000
1,olist_geolocation_dataset,geolocation_zip_code_prefix,1000163,1001.000,26530.000,36574.166,99990.000
2,olist_geolocation_dataset,geolocation_lat,1000163,-36.605,-22.919,-21.176,45.066
3,olist_geolocation_dataset,geolocation_lng,1000163,-101.467,-46.638,-46.391,121.105
4,olist_order_items_dataset,order_item_id,112650,1.000,1.000,1.198,21.000
5,olist_order_items_dataset,price,112650,0.850,74.990,120.654,6735.000
6,olist_order_items_dataset,freight_value,112650,0.000,16.260,19.990,409.680
7,olist_order_payments_dataset,payment_sequential,103886,1.000,1.000,1.093,29.000
8,olist_order_payments_dataset,payment_installments,103886,0.000,1.000,2.853,24.000
9,olist_order_payments_dataset,payment_value,103886,0.000,100.000,154.100,13664.080


In [4]:
orders = tables['olist_orders_dataset']
customers = tables['olist_customers_dataset']
items = tables['olist_order_items_dataset']
payments = tables['olist_order_payments_dataset']
reviews = tables['olist_order_reviews_dataset']
products = tables['olist_products_dataset']
sellers = tables['olist_sellers_dataset']
translation = tables['product_category_name_translation']
geo = tables['olist_geolocation_dataset']

def key_check(label, df, cols, must_be_unique=False):
    duplicate_rows = int(df.duplicated(cols, keep=False).sum())
    result = {'check': label, 'rows': len(df), 'unique_keys': int(df[cols].drop_duplicates().shape[0]), 'duplicate_key_rows': duplicate_rows, 'status': 'PASS' if duplicate_rows == 0 else 'REPORT'}
    print(result)
    if must_be_unique:
        assert duplicate_rows == 0, result
    return result

unmatched_category_rows = products.loc[products['product_category_name'].notna() & ~products['product_category_name'].isin(translation['product_category_name']), 'product_category_name']
unmatched_category_values = sorted(unmatched_category_rows.unique().tolist())
key_results = [
    key_check('orders.order_id', orders, ['order_id'], True),
    key_check('customers.customer_id', customers, ['customer_id'], True),
    key_check('order_items.order_id + order_item_id', items, ['order_id', 'order_item_id'], True),
    key_check('payments.order_id + payment_sequential', payments, ['order_id', 'payment_sequential'], True),
    key_check('products.product_id', products, ['product_id'], True),
    key_check('sellers.seller_id', sellers, ['seller_id'], True),
    key_check('category_translation.product_category_name', translation, ['product_category_name'], True),
    key_check('reviews.review_id', reviews, ['review_id']),
    key_check('reviews.order_id + review_id', reviews, ['order_id', 'review_id'], True),
    key_check('geolocation ZIP prefix', geo, ['geolocation_zip_code_prefix']),
]
print('Review IDs are not globally unique; the order_id + review_id combination is unique.')
print(f'Unmatched category rows: {int(unmatched_category_rows.size)}; distinct category values: {len(unmatched_category_values)}; values: {unmatched_category_values}')
print('Geolocation ZIP prefixes are not unique because multiple observations can share a prefix.')


{'check': 'orders.order_id', 'rows': 99441, 'unique_keys': 99441, 'duplicate_key_rows': 0, 'status': 'PASS'}
{'check': 'customers.customer_id', 'rows': 99441, 'unique_keys': 99441, 'duplicate_key_rows': 0, 'status': 'PASS'}


{'check': 'order_items.order_id + order_item_id', 'rows': 112650, 'unique_keys': 112650, 'duplicate_key_rows': 0, 'status': 'PASS'}
{'check': 'payments.order_id + payment_sequential', 'rows': 103886, 'unique_keys': 103886, 'duplicate_key_rows': 0, 'status': 'PASS'}
{'check': 'products.product_id', 'rows': 32951, 'unique_keys': 32951, 'duplicate_key_rows': 0, 'status': 'PASS'}


{'check': 'sellers.seller_id', 'rows': 3095, 'unique_keys': 3095, 'duplicate_key_rows': 0, 'status': 'PASS'}
{'check': 'category_translation.product_category_name', 'rows': 71, 'unique_keys': 71, 'duplicate_key_rows': 0, 'status': 'PASS'}
{'check': 'reviews.review_id', 'rows': 99224, 'unique_keys': 98410, 'duplicate_key_rows': 1603, 'status': 'REPORT'}


{'check': 'reviews.order_id + review_id', 'rows': 99224, 'unique_keys': 99224, 'duplicate_key_rows': 0, 'status': 'PASS'}
{'check': 'geolocation ZIP prefix', 'rows': 1000163, 'unique_keys': 19015, 'duplicate_key_rows': 999120, 'status': 'REPORT'}
Review IDs are not globally unique; the order_id + review_id combination is unique.
Unmatched category rows: 13; distinct category values: 2; values: ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']
Geolocation ZIP prefixes are not unique because multiple observations can share a prefix.


In [5]:
def unmatched(child, child_col, parent, parent_col):
    return int((~child[child_col].isin(parent[parent_col])).sum())

ri = pd.DataFrame([
    {'relationship': 'orders.customer_id -> customers.customer_id', 'unmatched_rows': unmatched(orders, 'customer_id', customers, 'customer_id')},
    {'relationship': 'order_items.order_id -> orders.order_id', 'unmatched_rows': unmatched(items, 'order_id', orders, 'order_id')},
    {'relationship': 'order_items.product_id -> products.product_id', 'unmatched_rows': unmatched(items, 'product_id', products, 'product_id')},
    {'relationship': 'order_items.seller_id -> sellers.seller_id', 'unmatched_rows': unmatched(items, 'seller_id', sellers, 'seller_id')},
    {'relationship': 'payments.order_id -> orders.order_id', 'unmatched_rows': unmatched(payments, 'order_id', orders, 'order_id')},
    {'relationship': 'reviews.order_id -> orders.order_id', 'unmatched_rows': unmatched(reviews, 'order_id', orders, 'order_id')},
    {'relationship': 'products.product_category_name -> category_translation.product_category_name', 'unmatched_rows': int((products['product_category_name'].notna() & ~products['product_category_name'].isin(translation['product_category_name'])).sum())},
])
display(ri)

items_with_category = items.merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
risk = pd.DataFrame([
    {'check': 'orders with multiple item rows', 'value': int((items.groupby('order_id').size() > 1).sum())},
    {'check': 'orders with multiple sellers', 'value': int((items.groupby('order_id')['seller_id'].nunique() > 1).sum())},
    {'check': 'orders with multiple product categories', 'value': int((items_with_category.groupby('order_id')['product_category_name'].nunique() > 1).sum())},
    {'check': 'maximum items per order', 'value': int(items.groupby('order_id').size().max())},
    {'check': 'maximum sellers per order', 'value': int(items.groupby('order_id')['seller_id'].nunique().max())},
    {'check': 'maximum categories per order', 'value': int(items_with_category.groupby('order_id')['product_category_name'].nunique().max())},
    {'check': 'orders missing item rows', 'value': int((~orders['order_id'].isin(items['order_id'])).sum())},
    {'check': 'orders with multiple payments', 'value': int((payments.groupby('order_id').size() > 1).sum())},
    {'check': 'orders with multiple reviews', 'value': int((reviews.groupby('order_id').size() > 1).sum())},
])
display(risk)


,relationship,unmatched_rows
0,orders.customer_id -> customers.customer_id,0
1,order_items.order_id -> orders.order_id,0
2,order_items.product_id -> products.product_id,0
3,order_items.seller_id -> sellers.seller_id,0
4,payments.order_id -> orders.order_id,0
5,reviews.order_id -> orders.order_id,0
6,products.product_category_name -> category_tra...,13


,check,value
0,orders with multiple item rows,9803
1,orders with multiple sellers,1278
2,orders with multiple product categories,727
3,maximum items per order,21
4,maximum sellers per order,5
5,maximum categories per order,3
6,orders missing item rows,775
7,orders with multiple payments,2961
8,orders with multiple reviews,547


In [6]:
orders_dates = orders.copy()
for col in date_columns(orders_dates):
    orders_dates[col] = pd.to_datetime(orders_dates[col], errors='coerce')
items_dates = items.copy()
items_dates['shipping_limit_date'] = pd.to_datetime(items_dates['shipping_limit_date'], errors='coerce')
activity_start = orders_dates['order_purchase_timestamp'].min()
activity_end = orders_dates['order_estimated_delivery_date'].max()

outside_rows = []
for col in date_columns(orders_dates):
    s = orders_dates[col].dropna()
    outside_rows.append({'table': 'orders', 'column': col, 'before_activity_start': int((s < activity_start).sum()), 'after_activity_end': int((s > activity_end).sum())})
outside_rows.append({'table': 'order_items', 'column': 'shipping_limit_date', 'before_activity_start': int((items_dates['shipping_limit_date'] < activity_start).sum()), 'after_activity_end': int((items_dates['shipping_limit_date'] > activity_end).sum())})
print(f'Normal order-activity window: {activity_start} to {activity_end}')
display(pd.DataFrame(outside_rows))
print(f'shipping_limit_date after 2018-12-31: {int((items_dates["shipping_limit_date"] > pd.Timestamp("2018-12-31 23:59:59")).sum())}')

sequence_checks = pd.DataFrame([
    {'check': 'approval before purchase', 'violations': int((orders_dates['order_approved_at'] < orders_dates['order_purchase_timestamp']).sum())},
    {'check': 'carrier handoff before approval', 'violations': int((orders_dates['order_delivered_carrier_date'] < orders_dates['order_approved_at']).sum())},
    {'check': 'customer delivery before carrier handoff', 'violations': int((orders_dates['order_delivered_customer_date'] < orders_dates['order_delivered_carrier_date']).sum())},
])
display(sequence_checks)
print('Anomalies are reported only; no rows are deleted or corrected.')


Normal order-activity window: 2016-09-04 21:15:19 to 2018-11-12 00:00:00


,table,column,before_activity_start,after_activity_end
0,orders,order_purchase_timestamp,0,0
1,orders,order_approved_at,0,0
2,orders,order_delivered_carrier_date,0,0
3,orders,order_delivered_customer_date,0,0
4,orders,order_estimated_delivery_date,0,0
5,order_items,shipping_limit_date,0,4


shipping_limit_date after 2018-12-31: 4


,check,violations
0,approval before purchase,0
1,carrier handoff before approval,1359
2,customer delivery before carrier handoff,23


Anomalies are reported only; no rows are deleted or corrected.


In [7]:
def md_table(headers, rows):
    lines = ['| ' + ' | '.join(headers) + ' |', '| ' + ' | '.join(['---'] * len(headers)) + ' |']
    lines.extend('| ' + ' | '.join(str(value) for value in row) + ' |' for row in rows)
    return '\n'.join(lines)

grain_labels = {
    'olist_orders_dataset': 'One row per order',
    'olist_customers_dataset': 'One row per order-level customer record',
    'olist_order_items_dataset': 'One row per order_id + order_item_id',
    'olist_order_payments_dataset': 'One row per order_id + payment_sequential',
    'olist_order_reviews_dataset': 'One row per review record',
    'olist_products_dataset': 'One row per product',
    'olist_sellers_dataset': 'One row per seller',
    'product_category_name_translation': 'One row per Portuguese category',
    'olist_geolocation_dataset': 'Multiple observations per ZIP prefix',
}
inventory_rows = [[name + '.csv', int(row['rows']), grain_labels[name]] for name, row in profile.iterrows()]
missing_lines = []
for name, df in tables.items():
    for col, count in df.isna().sum().items():
        if count:
            missing_lines.append(f'- `{name}.{col}`: {int(count):,} ({df[col].isna().mean() * 100:.2f}%)')

date_rows = []
for row in date_range_rows:
    date_rows.append([row['table'] + '.' + row['column'], str(row['min']), str(row['max']), int(row['missing'])])
ri_rows = [[row['relationship'], int(row['unmatched_rows'])] for row in ri.to_dict('records')]
risk_rows = [[row['check'], int(row['value'])] for row in risk.to_dict('records')]
sequence_rows = [[row['check'], int(row['violations'])] for row in sequence_checks.to_dict('records')]
shipping_after_2018 = int((items_dates['shipping_limit_date'] > pd.Timestamp('2018-12-31 23:59:59')).sum())
category_values_text = ', '.join(f'`{value}`' for value in unmatched_category_values)
report_sections = [
    '# Generated data quality report',
    '',
    'This report was generated by `python/01_data_profiling.ipynb` from the current CSV files in `data/raw`. It is deterministic and contains no execution timestamp.',
    'No raw CSV files were modified, corrected, deleted, or overwritten.',
    '',
    '## Dataset inventory',
    '',
    md_table(['Table', 'Rows', 'Grain'], inventory_rows),
    '',
    '## Table grains and keys',
    '',
    '- `order_id` is unique in orders.',
    '- `customer_id` is unique in customers.',
    '- `order_id` + `order_item_id` is unique in order items.',
    '- `order_id` + `payment_sequential` is unique in payments.',
    '- `product_id` is unique in products.',
    '- `seller_id` is unique in sellers.',
    '- `product_category_name` is unique in the category translation table.',
    '- `review_id` is not globally unique; `order_id` + `review_id` is unique in this extract.',
    '- Geolocation ZIP prefixes are not unique.',
    '',
    '## Missing-data summary',
    '',
    '\n'.join(missing_lines) if missing_lines else 'No missing values found.',
    '',
    '## Referential-integrity results',
    '',
    md_table(['Relationship', 'Unmatched rows'], ri_rows),
    '',
    f'- {int(unmatched_category_rows.size):,} product rows have non-null categories without English translations. Those rows represent {len(unmatched_category_values)} distinct category values: {category_values_text}.',
    '',
    '## One-to-many relationship risks',
    '',
    md_table(['Check', 'Value'], risk_rows),
    '',
    '## Date ranges and anomalies',
    '',
    md_table(['Date field', 'Minimum', 'Maximum', 'Missing'], date_rows),
    '',
    f'- Normal order-activity window: {activity_start} through {activity_end}.',
    f'- `shipping_limit_date` values after 2018-12-31: {shipping_after_2018:,}.',
    '',
    md_table(['Lifecycle check', 'Violations'], sequence_rows),
    '',
    'Anomalies are reported only. No raw rows are deleted or corrected.',
    '',
    '## Modeling implications',
    '',
    'Use separate native-grain tables: `fact_orders` (one row per order), `fact_order_items` (one row per order item), `fact_payments` (one row per payment), and `fact_reviews` (one row per review). Aggregate each fact to `order_id` before combining order-level metrics.',
    '',
    '## Known limitations',
    '',
    f'- {int(products["product_category_name"].isna().sum()):,} products have no category.',
    '- Geolocation contains repeated observations and requires an explicit aggregation rule.',
    '- Review identifiers are not globally unique.',
    '- This is a data-quality and modeling audit, not a causal analysis.',
]
report_path = REPO_ROOT / 'docs' / 'data-quality-report.md'
report_path.write_text('\n'.join(report_sections) + '\n', encoding='utf-8')
print('Generated report: docs/data-quality-report.md')


Generated report: docs/data-quality-report.md
